# Encoder Stack, Decoder Stack, and Masked Attention

**Module 13** — Self-Attention & the Transformer

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Now let's see the full Transformer architecture in PyTorch. This is the complete encoder-decoder model:


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        B, T, _ = Q.shape
        # Project and reshape: (B, T, d_model) -> (B, h, T, d_k)
        Q = self.W_Q(Q).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V  # (B, h, T, d_k)
        
        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_O(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention + residual + norm
        attn_out = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

# Usage
d_model, n_heads, d_ff, N = 512, 8, 2048, 6
encoder = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(N)])

### Step 2: Forward pass through encoder stack


In [ ]:
x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
for block in encoder:
    x = block(x)
print(f"Output shape: {x.shape}")  # (2, 10, 512) — same as input!


---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
